**Bigger-font variant (2026-08 supervisor feedback item 2).** Copied from the original simple-plot notebook, not edited in place — same convention as `100_versions_pie_plot_simple_bigfont.ipynb` and the original-vs-simple-plot split before it. Reads from the bigfont chart source (`100_pie_charts_simple_bigfont/`) and writes to a separate `*_bigfont` post/output tree throughout, so nothing here collides with the existing simple-plot pilot data or its results. See `docs/SESSION_HANDOFF.md` for context.

---
# HTML to PNG converter
Run using virtual environment! Python 3.12 End 2 correct
This code works here because I did the following and there was already an existing version of chromium donwloaded on the server. So it was enough for me to download the web driver in the venv to get it to work. Since gpu server has no chromium, cannot download it, cant use web driver.

In [1]:
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
#from webdriver_manager.core.utils import ChromeType
#from webdriver_manager.chrome import ChromeDriverManager
#from webdriver_manager.core.os_manager import ChromeType
#from webdriver_manager.chrome import ChromeDriverManager


In [2]:
import os

# Only move up if we're still in the utils/ directory
if os.getcwd().endswith('/utils'):
    os.chdir('..')

print(os.getcwd())  # should always show conformity-llms-facebook-posts

/home/scc/miranda.barros-everett/conformity-llms-facebook-posts


## Converting whole folders

In [4]:
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import os
import time

def capture_screenshot(html_path, output_image="screenshot.png"):
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=800,1200")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.binary_location = "/usr/bin/chromium"

    # Selenium Manager (built into Selenium 4.6+) auto-detects the installed Chromium binary
    # (via chrome_options.binary_location above) and fetches a matching ChromeDriver version --
    # avoids the version-mismatch break that happens whenever Chromium auto-updates itself but
    # a manually-downloaded driver at a fixed path (~/chromedriver) does not.
    driver = webdriver.Chrome(options=chrome_options)

    try:
        file_url = "file://" + os.path.abspath(html_path)
        driver.get(file_url)
        time.sleep(2)
        driver.save_screenshot(output_image)
        print(f"Screenshot saved as {output_image}")
    finally:
        driver.quit()


def process_folder(label: str, html_dir: Path, png_dir: Path):
    """Process all HTML files in a folder, saving PNGs to the output dir."""
    if not html_dir.exists():
        print(f"[SKIP] Directory not found: {html_dir}")
        return

    html_files = sorted(html_dir.glob("*.html"))
    if not html_files:
        print(f"[SKIP] No HTML files found in: {html_dir}")
        return

    png_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n[{label.upper()}] Processing {len(html_files)} file(s) from {html_dir}")
    '''
    for html_file in html_files:
        output_png = png_dir / (html_file.stem + ".png")

        if output_png.exists():
            print(f"  [SKIP] Already exists: {output_png.name}")
            continue

        print(f"  → {html_file.name}")
        capture_screenshot(str(html_file), str(output_png))
    '''
    for idx, html_file in enumerate(html_files):
        output_png = png_dir / (html_file.stem + ".png")
        
        if output_png.exists():
            print(f"  [SKIP] Already exists: {output_png.name}")
            continue
        
        print(f"  → {html_file.name}")
        capture_screenshot(str(html_file), str(output_png))
        
        # pause every 10 files
        if idx % 10 == 0:
            time.sleep(3)

BASE_DIR = Path().resolve()
METRICS_DIR = BASE_DIR / "spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford"
REACTION_VALUES = [10, 100, 1000, 10000, 100000, 1000000]

# Covers all three metrics conditions in one pass -- process_folder's own skip-logic
# (skip if the PNG already exists) means this safely resumes each condition from wherever
# it left off, rather than needing the block below hand-edited and re-run per condition.
CONDITIONS = ["likes_only_noise", "realistic"]

for condition in CONDITIONS:
    for label in ("correct", "incorrect"):
        for scale_value in REACTION_VALUES:
            process_folder(
                label=f"{condition}/{label}/{scale_value}",
                html_dir=METRICS_DIR / label / "html" / condition / str(scale_value),
                png_dir=METRICS_DIR / label / "PNGs" / condition / str(scale_value),
            )

print("\nDone.")


[LIKES_ONLY_NOISE/CORRECT/10] Processing 100 file(s) from /home/scc/miranda.barros-everett/conformity-llms-facebook-posts/spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/html/likes_only_noise/10
  [SKIP] Already exists: 001_remy_ashford_c.png
  [SKIP] Already exists: 002_remy_ashford_c.png
  [SKIP] Already exists: 003_remy_ashford_c.png
  [SKIP] Already exists: 004_remy_ashford_c.png
  [SKIP] Already exists: 005_remy_ashford_c.png
  [SKIP] Already exists: 006_remy_ashford_c.png
  [SKIP] Already exists: 007_remy_ashford_c.png
  [SKIP] Already exists: 008_remy_ashford_c.png
  [SKIP] Already exists: 009_remy_ashford_c.png
  → 010_remy_ashford_c.html
Screenshot saved as /home/scc/miranda.barros-everett/conformity-llms-facebook-posts/spotify_pie_plot/pie_plot_posts/metrics_simple_plot_bigfont/remy-ashford/correct/PNGs/likes_only_noise/10/010_remy_ashford_c.png
  → 011_remy_ashford_c.html
Screenshot saved as /home/scc/miranda.barros-everett/conformity-llms-f

WebDriverException: Message: Service /home/scc/miranda.barros-everett/.cache/selenium/chromedriver/linux64/150.0.7871.124/chromedriver unexpectedly exited. Status code was: 1


In [ ]:
BASELINE_DIR = BASE_DIR / "spotify_pie_plot/pie_plot_posts/baselines_simple_plot_bigfont"

for label in ("correct", "incorrect"):
    # remy-ashford baseline files sit directly in html/ (no subfolder)
    process_folder(
        label=f"{label}/remy-ashford",
        html_dir=BASELINE_DIR / label / "html",
        png_dir=BASELINE_DIR / label / "PNGs",
    )
    # news-outlet baseline files sit in a "news" subfolder
    '''
    process_folder(
        label=f"{label}/news",
        html_dir=BASELINE_DIR / label / "html" / "news",
        png_dir=BASELINE_DIR / label / "PNGs" / "news",
    )
    '''

print("\nDone.")